## Import Libraries

In [7]:
import os
import json
import joblib
import warnings
import mlflow

import numpy as np
import pandas as pd

from pathlib import Path

warnings.filterwarnings("ignore")

print("Libraries imported successfully.")

Libraries imported successfully.


In [8]:
# Main project directory (parent of Notebooks)
PROJECT_DIR = Path.cwd().parent

# Project folders
MODEL_DIR = PROJECT_DIR / "models"
OUTPUT_DIR = PROJECT_DIR / "outputs"

# MLflow exported files directory
MLFLOW_INPUT_DIR = MODEL_DIR / "mlflow_inputs"

# Create directories if they do not exist
MODEL_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project Directory:", PROJECT_DIR)
print("Model Directory:", MODEL_DIR)
print("Output Directory:", OUTPUT_DIR)
print("MLflow Input Directory:", MLFLOW_INPUT_DIR)
print("MLflow Input Directory Exists:", MLFLOW_INPUT_DIR.exists())

Project Directory: C:\Users\ijish\Desktop\Rossmann Sales Forecasting Project
Model Directory: C:\Users\ijish\Desktop\Rossmann Sales Forecasting Project\models
Output Directory: C:\Users\ijish\Desktop\Rossmann Sales Forecasting Project\outputs
MLflow Input Directory: C:\Users\ijish\Desktop\Rossmann Sales Forecasting Project\models\mlflow_inputs
MLflow Input Directory Exists: True


In [9]:
# SQLite database for MLflow tracking
MLFLOW_DB_PATH = PROJECT_DIR / "mlflow.db"

mlflow.set_tracking_uri(
    f"sqlite:///{MLFLOW_DB_PATH.as_posix()}"
)

print("MLflow Tracking URI:", mlflow.get_tracking_uri())
print("MLflow SQLite database:", MLFLOW_DB_PATH)

MLflow Tracking URI: sqlite:///C:/Users/ijish/Desktop/Rossmann Sales Forecasting Project/mlflow.db
MLflow SQLite database: C:\Users\ijish\Desktop\Rossmann Sales Forecasting Project\mlflow.db


In [10]:
experiments = [
    "Rossmann_Sales_Prediction",
    "Rossmann_Customers_Prediction",
    "Rossmann_LSTM_Forecasting"
]

for experiment_name in experiments:
    mlflow.set_experiment(experiment_name)

print("All MLflow experiments created successfully!")

All MLflow experiments created successfully!


In [11]:
experiments_df = mlflow.search_experiments()

for exp in experiments_df:

    print("Experiment Name:", exp.name)
    print("Experiment ID:", exp.experiment_id)
    print("Lifecycle Stage:", exp.lifecycle_stage)
    print("-" * 50)

Experiment Name: Rossmann_LSTM_Forecasting
Experiment ID: 3
Lifecycle Stage: active
--------------------------------------------------
Experiment Name: Rossmann_Customers_Prediction
Experiment ID: 2
Lifecycle Stage: active
--------------------------------------------------
Experiment Name: Rossmann_Sales_Prediction
Experiment ID: 1
Lifecycle Stage: active
--------------------------------------------------
Experiment Name: Default
Experiment ID: 0
Lifecycle Stage: active
--------------------------------------------------


### Exported Model Files

In [12]:
expected_files = [
    "sales_candidate_models.pkl",
    "customers_candidate_models.pkl",
    "sales_validation_results.csv",
    "sales_open_store_results.csv",
    "customers_validation_results.csv",
    "customers_open_store_results.csv"
]

for file in expected_files:

    file_path = MLFLOW_INPUT_DIR / file

    if file_path.exists():
        print("Found:", file)
    else:
        print("Missing:", file)

Found: sales_candidate_models.pkl
Found: customers_candidate_models.pkl
Found: sales_validation_results.csv
Found: sales_open_store_results.csv
Found: customers_validation_results.csv
Found: customers_open_store_results.csv


In [13]:
# Load candidate models
trained_models = joblib.load(
    MLFLOW_INPUT_DIR / "sales_candidate_models.pkl"
)

trained_customer_models = joblib.load(
    MLFLOW_INPUT_DIR / "customers_candidate_models.pkl"
)

# Load evaluation results
results_df = pd.read_csv(
    MLFLOW_INPUT_DIR / "sales_validation_results.csv"
)

open_results_df = pd.read_csv(
    MLFLOW_INPUT_DIR / "sales_open_store_results.csv"
)

customer_results_df = pd.read_csv(
    MLFLOW_INPUT_DIR / "customers_validation_results.csv"
)

customer_open_results_df = pd.read_csv(
    MLFLOW_INPUT_DIR / "customers_open_store_results.csv"
)

print("Sales candidate models:", trained_models.keys())
print("Customers candidate models:", trained_customer_models.keys())

display(results_df)
display(customer_results_df)

Sales candidate models: dict_keys(['Linear Regression', 'Decision Tree', 'Random Forest', 'Gradient Boosting'])
Customers candidate models: dict_keys(['Linear Regression', 'Decision Tree', 'Random Forest', 'Gradient Boosting'])


,Model,MAE,RMSE,R2,RMSPE
0,Decision Tree,917.938418,1455.957901,0.846069,22.698893
1,Random Forest,1387.855690,2062.698646,0.691042,36.162540
2,Gradient Boosting,1749.439770,2547.381163,0.528789,46.744172
3,Linear Regression,1674.222930,2430.040813,0.571200,47.526624


,Model,MAE,RMSE,R2,RMSPE
0,Decision Tree,72.409907,126.443315,0.916803,20.891682
1,Random Forest,131.246996,191.628034,0.808912,34.338456
2,Linear Regression,188.815098,285.257388,0.576563,48.860868
3,Gradient Boosting,197.483007,293.628817,0.551345,49.774219


## Log Sales Models

In [14]:
print("MLflow Version:", mlflow.__version__)

MLflow Version: 3.18.0


### Log Sales Candidate Models

In [15]:
mlflow.set_experiment("Rossmann_Sales_Prediction")

for model_name, model in trained_models.items():

    with mlflow.start_run(run_name=model_name):

        # Log model information
        mlflow.log_param("model_name", model_name)
        mlflow.log_param("target", "Sales")

        # Get evaluation results
        row = results_df[
            results_df["Model"] == model_name
        ]

        if not row.empty:

            for metric in ["MAE", "RMSE", "R2", "RMSPE"]:

                if metric in row.columns:
                    mlflow.log_metric(
                        metric,
                        float(row.iloc[0][metric])
                    )

        # Log model using Cloudpickle
        mlflow.sklearn.log_model(
            sk_model=model,
            name="sales_model",
            serialization_format="cloudpickle"
        )

        print(f"Successfully logged: {model_name}")

print("All Sales candidate models logged successfully!")

2026/11/13 20:37:50 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet

2026/11/13 20:37:50 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably

Successfully logged: Linear Regression


2026/11/13 20:38:18 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Successfully logged: Decision Tree


2026/11/13 20:38:29 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Successfully logged: Random Forest


2026/11/13 20:38:42 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Successfully logged: Gradient Boosting
All Sales candidate models logged successfully!


### Log Sales Open-Store Evaluation Results

In [16]:
mlflow.set_experiment("Rossmann_Sales_Prediction")

for _, row in open_results_df.iterrows():

    model_name = str(row["Model"])

    with mlflow.start_run(
        run_name=f"Open_Store_{model_name}"
    ):

        mlflow.log_param("model_name", model_name)
        mlflow.log_param("target", "Sales")
        mlflow.log_param("evaluation_type", "Open Stores Only")

        for metric in ["MAE", "RMSE", "R2", "RMSPE"]:

            if metric in open_results_df.columns:

                value = row[metric]

                if pd.notna(value):

                    mlflow.log_metric(
                        metric,
                        float(value)
                    )

        print(f"Logged open-store evaluation: {model_name}")

print("Sales open-store evaluation completed!")

Logged open-store evaluation: Linear Regression
Logged open-store evaluation: Decision Tree
Logged open-store evaluation: Random Forest
Logged open-store evaluation: Gradient Boosting
Sales open-store evaluation completed!


## Log Customers Models

### Log Customers Candidate Models

In [17]:
mlflow.set_experiment("Rossmann_Customers_Prediction")

for model_name, model in trained_customer_models.items():

    with mlflow.start_run(run_name=model_name):

        mlflow.log_param("model_name", model_name)
        mlflow.log_param("target", "Customers")

        row = customer_results_df[
            customer_results_df["Model"] == model_name
        ]

        if not row.empty:

            for metric in ["MAE", "RMSE", "R2", "RMSPE"]:

                if metric in row.columns:
                    mlflow.log_metric(
                        metric,
                        float(row.iloc[0][metric])
                    )

        mlflow.sklearn.log_model(
            sk_model=model,
            name="sales_model",
            serialization_format="cloudpickle"
        )

        print("Logged:", model_name)

print("Customers candidate models logged successfully!")

2026/11/13 20:38:55 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Logged: Linear Regression


2026/11/13 20:39:06 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Logged: Decision Tree


2026/11/13 20:39:18 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Logged: Random Forest


2026/11/13 20:39:30 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Logged: Gradient Boosting
Customers candidate models logged successfully!


### Log Customers Open-Store Evaluation

In [18]:
mlflow.set_experiment("Rossmann_Customers_Prediction")

for _, row in customer_open_results_df.iterrows():

    model_name = str(row["Model"])

    with mlflow.start_run(
        run_name=f"Open_Store_{model_name}"
    ):

        mlflow.log_param("model_name", model_name)
        mlflow.log_param("target", "Customers")
        mlflow.log_param("evaluation_type", "Open Stores Only")

        for metric in ["MAE", "RMSE", "R2", "RMSPE"]:

            if metric in customer_open_results_df.columns:

                value = row[metric]

                if pd.notna(value):

                    mlflow.log_metric(
                        metric,
                        float(value)
                    )

        print(f"Logged open-store evaluation: {model_name}")

print("Customers open-store evaluation completed!")

Logged open-store evaluation: Linear Regression
Logged open-store evaluation: Decision Tree
Logged open-store evaluation: Random Forest
Logged open-store evaluation: Gradient Boosting
Customers open-store evaluation completed!


## Log the Final Models

In [19]:
# Find saved final model files
sales_model_files = sorted(
    MODEL_DIR.glob("rossmann_sales_model_*.pkl"),
    key=lambda x: x.stat().st_mtime,
    reverse=True
)

customer_model_files = sorted(
    MODEL_DIR.glob("rossmann_customers_model_*.pkl"),
    key=lambda x: x.stat().st_mtime,
    reverse=True
)

if not sales_model_files:
    raise FileNotFoundError(
        "No saved Sales model found in the models folder."
    )

if not customer_model_files:
    raise FileNotFoundError(
        "No saved Customers model found in the models folder."
    )

# Latest saved models
sales_model_path = sales_model_files[0]
customers_model_path = customer_model_files[0]

# Load final models
final_sales_model = joblib.load(sales_model_path)
final_customers_model = joblib.load(customers_model_path)

print("Final Sales Model:", sales_model_path.name)
print("Final Customers Model:", customers_model_path.name)

Final Sales Model: rossmann_sales_model_12-11-2026-22-49-38.pkl
Final Customers Model: rossmann_customers_model_12-11-2026-22-53-37.pkl


### Log the Final Sales Model

In [20]:
mlflow.set_experiment("Rossmann_Sales_Prediction")

# Identify the selected model using validation results
sales_best_model_name = (
    results_df.sort_values("MAE").iloc[0]["Model"]
)

sales_best_row = results_df[
    results_df["Model"] == sales_best_model_name
].iloc[0]

with mlflow.start_run(
    run_name=f"Final_Sales_{sales_best_model_name}"
):

    # Parameters
    mlflow.log_param("model_name", sales_best_model_name)
    mlflow.log_param("target", "Sales")
    mlflow.log_param("model_type", "Final Selected Model")
    mlflow.log_param("model_file", sales_model_path.name)
    mlflow.log_param("training_data", "Complete training dataset")

    # Validation metrics
    for metric in ["MAE", "RMSE", "R2", "RMSPE"]:

        if metric in sales_best_row.index:
            value = sales_best_row[metric]

            if pd.notna(value):
                mlflow.log_metric(
                    metric,
                    float(value)
                )

    # Log final sklearn pipeline
    mlflow.sklearn.log_model(
        sk_model=final_sales_model,
        name="final_sales_model",
        serialization_format="cloudpickle"
    )

    # Log original saved model file
    mlflow.log_artifact(
        str(sales_model_path),
        artifact_path="saved_model_file"
    )

    print("Final Sales model logged successfully!")
    print("Selected Model:", sales_best_model_name)

2026/11/13 20:39:43 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Final Sales model logged successfully!
Selected Model: Decision Tree


### Log the Final Customers Model

In [21]:
mlflow.set_experiment("Rossmann_Customers_Prediction")

# Identify the selected model using validation results
customers_best_model_name = (
    customer_results_df.sort_values("MAE").iloc[0]["Model"]
)

customers_best_row = customer_results_df[
    customer_results_df["Model"] == customers_best_model_name
].iloc[0]

with mlflow.start_run(
    run_name=f"Final_Customers_{customers_best_model_name}"
):

    # Parameters
    mlflow.log_param("model_name", customers_best_model_name)
    mlflow.log_param("target", "Customers")
    mlflow.log_param("model_type", "Final Selected Model")
    mlflow.log_param("model_file", customers_model_path.name)
    mlflow.log_param("training_data", "Complete training dataset")

    # Validation metrics
    for metric in ["MAE", "RMSE", "R2", "RMSPE"]:

        if metric in customers_best_row.index:
            value = customers_best_row[metric]

            if pd.notna(value):
                mlflow.log_metric(
                    metric,
                    float(value)
                )

    # Log final sklearn pipeline
    mlflow.sklearn.log_model(
        sk_model=final_customers_model,
        name="final_customers_model",
        serialization_format="cloudpickle"
    )

    # Log original saved model file
    mlflow.log_artifact(
        str(customers_model_path),
        artifact_path="saved_model_file"
    )

    print("Final Customers model logged successfully!")
    print("Selected Model:", customers_best_model_name)

2026/11/13 20:39:57 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Final Customers model logged successfully!
Selected Model: Decision Tree


In [22]:
for experiment_name in [
    "Rossmann_Sales_Prediction",
    "Rossmann_Customers_Prediction"
]:

    print("\n", experiment_name)

    runs = mlflow.search_runs(
        experiment_names=[experiment_name]
    )

    display(
        runs[
            [
                "tags.mlflow.runName",
                "metrics.MAE",
                "metrics.RMSE",
                "metrics.R2",
                "metrics.RMSPE",
                "status"
            ]
        ]
    )


 Rossmann_Sales_Prediction


,tags.mlflow.runName,metrics.MAE,metrics.RMSE,metrics.R2,metrics.RMSPE,status
0,Final_Sales_Decision Tree,917.938418,1455.957901,0.846069,22.698893,FINISHED
1,Open_Store_Gradient Boosting,1932.091407,2731.222328,0.194953,46.744172,FINISHED
2,Open_Store_Random Forest,1602.423881,2218.550703,0.468815,36.162540,FINISHED
3,Open_Store_Decision Tree,1063.159664,1566.900034,0.735034,22.698893,FINISHED
4,Open_Store_Linear Regression,1908.019040,2611.200113,0.264153,47.526624,FINISHED
5,Gradient Boosting,1749.439770,2547.381163,0.528789,46.744172,FINISHED
6,Random Forest,1387.855690,2062.698646,0.691042,36.162540,FINISHED
7,Decision Tree,917.938418,1455.957901,0.846069,22.698893,FINISHED
8,Linear Regression,1674.222930,2430.040813,0.571200,47.526624,FINISHED
9,Open_Store_Gradient Boosting,1932.091407,2731.222328,0.194953,46.744172,FINISHED



 Rossmann_Customers_Prediction


,tags.mlflow.runName,metrics.MAE,metrics.RMSE,metrics.R2,metrics.RMSPE,status
0,Final_Customers_Decision Tree,72.409907,126.443315,0.916803,20.891682,FINISHED
1,Open_Store_Gradient Boosting,214.254359,313.825687,0.338659,NaN,FINISHED
2,Open_Store_Random Forest,151.519253,206.154509,0.714613,NaN,FINISHED
3,Open_Store_Decision Tree,83.865422,136.078134,0.875656,NaN,FINISHED
4,Open_Store_Linear Regression,212.800041,305.922477,0.371549,NaN,FINISHED
5,Gradient Boosting,197.483007,293.628817,0.551345,49.774219,FINISHED
6,Random Forest,131.246996,191.628034,0.808912,34.338456,FINISHED
7,Decision Tree,72.409907,126.443315,0.916803,20.891682,FINISHED
8,Linear Regression,188.815098,285.257388,0.576563,48.860868,FINISHED
9,Open_Store_Gradient Boosting,214.254359,313.825687,0.338659,NaN,FINISHED


## LSTM Model Tracking

In [23]:
LSTM_INPUT_DIR = MLFLOW_INPUT_DIR

print("LSTM Input Directory:", LSTM_INPUT_DIR)

print("\nAvailable files:")

for file in sorted(LSTM_INPUT_DIR.iterdir()):
    if file.is_file():
        print(file.name)

LSTM Input Directory: C:\Users\ijish\Desktop\Rossmann Sales Forecasting Project\models\mlflow_inputs

Available files:
customers_candidate_models.pkl
customers_open_store_results.csv
customers_validation_results.csv
lstm_validation_metrics.json
mlflow_lstm_config.json
rossmann_lstm_multivariate.keras
rossmann_lstm_X_scaler.pkl
rossmann_lstm_y_scaler.pkl
sales_candidate_models.pkl
sales_open_store_results.csv
sales_validation_results.csv
split_info.json


In [24]:
# Required LSTM files
lstm_files = {
    "model": "rossmann_lstm_multivariate.keras",
    "x_scaler": "rossmann_lstm_X_scaler.pkl",
    "y_scaler": "rossmann_lstm_y_scaler.pkl",
    "config": "mlflow_lstm_config.json",
    "metrics": "lstm_validation_metrics.json"
}

# Locate files, including in subfolders
found_files = {}

for key, filename in lstm_files.items():

    matches = list(LSTM_INPUT_DIR.rglob(filename))

    if not matches:
        raise FileNotFoundError(
            f"Missing file: {filename}"
        )

    found_files[key] = matches[0]

    print(f"{key}: {matches[0]}")

print("\nAll required LSTM files found successfully!")

model: C:\Users\ijish\Desktop\Rossmann Sales Forecasting Project\models\mlflow_inputs\rossmann_lstm_multivariate.keras
x_scaler: C:\Users\ijish\Desktop\Rossmann Sales Forecasting Project\models\mlflow_inputs\rossmann_lstm_X_scaler.pkl
y_scaler: C:\Users\ijish\Desktop\Rossmann Sales Forecasting Project\models\mlflow_inputs\rossmann_lstm_y_scaler.pkl
config: C:\Users\ijish\Desktop\Rossmann Sales Forecasting Project\models\mlflow_inputs\mlflow_lstm_config.json
metrics: C:\Users\ijish\Desktop\Rossmann Sales Forecasting Project\models\mlflow_inputs\lstm_validation_metrics.json

All required LSTM files found successfully!


### Load LSTM Configuration and Validation Metrics

In [25]:
# Load LSTM configuration
with open(found_files["config"], "r") as f:
    lstm_config = json.load(f)

# Load LSTM validation metrics
with open(found_files["metrics"], "r") as f:
    lstm_metrics = json.load(f)

print("LSTM Configuration:")
display(pd.DataFrame(
    lstm_config.items(),
    columns=["Parameter", "Value"]
))

print("\nLSTM Validation Metrics:")
display(pd.DataFrame(lstm_metrics).T)

LSTM Configuration:


,Parameter,Value
0,model_type,Multivariate LSTM
1,lookback,30
2,features,"[Sales, OpenStores, PromoStores, SchoolHoliday..."
3,target,Sales
4,scaling_range,"[-1, 1]"
5,forecast_level,Daily aggregate sales across all stores
6,forecast_method,Recursive with known future covariates



LSTM Validation Metrics:


,MAE,RMSE,R2
rolling_one_step,843193.42,1470030.87,0.7927
recursive,857818.27,1475753.09,0.7911


### Log the Final LSTM Model

In [26]:
# Select LSTM experiment
mlflow.set_experiment(
    "Rossmann_LSTM_Forecasting"
)

with mlflow.start_run(
    run_name="Final_Multivariate_LSTM"
):

    # --------------------------------
    # 1. LOG MODEL PARAMETERS
    # --------------------------------

    mlflow.log_param(
        "model_type",
        "Multivariate LSTM"
    )

    mlflow.log_param(
        "target",
        "Daily Aggregate Sales"
    )

    mlflow.log_param(
        "forecast_level",
        "All Stores Combined"
    )

    mlflow.log_param(
        "lookback",
        lstm_config["lookback"]
    )

    mlflow.log_param(
        "number_of_features",
        len(lstm_config["features"])
    )

    mlflow.log_param(
        "features",
        ", ".join(lstm_config["features"])
    )

    mlflow.log_param(
        "forecast_method",
        lstm_config["forecast_method"]
    )

    mlflow.log_param(
        "scaling_range",
        str(lstm_config["scaling_range"])
    )

    # --------------------------------
    # 2. LOG ROLLING ONE-STEP METRICS
    # --------------------------------

    for metric, value in lstm_metrics["rolling_one_step"].items():

        mlflow.log_metric(
            f"rolling_one_step_{metric.lower()}",
            float(value)
        )

    # --------------------------------
    # 3. LOG RECURSIVE METRICS
    # --------------------------------

    for metric, value in lstm_metrics["recursive"].items():

        mlflow.log_metric(
            f"recursive_{metric.lower()}",
            float(value)
        )

    # --------------------------------
    # 4. LOG MODEL AND SCALERS
    # --------------------------------

    mlflow.log_artifact(
        str(found_files["model"]),
        artifact_path="model"
    )

    mlflow.log_artifact(
        str(found_files["x_scaler"]),
        artifact_path="preprocessing"
    )

    mlflow.log_artifact(
        str(found_files["y_scaler"]),
        artifact_path="preprocessing"
    )

    # --------------------------------
    # 5. LOG CONFIGURATION AND METRICS
    # --------------------------------

    mlflow.log_artifact(
        str(found_files["config"]),
        artifact_path="configuration"
    )

    mlflow.log_artifact(
        str(found_files["metrics"]),
        artifact_path="evaluation"
    )

    print("LSTM model logged successfully!")

print("LSTM MLflow tracking completed!")

LSTM model logged successfully!
LSTM MLflow tracking completed!


In [27]:
mlflow.set_experiment(
    "Rossmann_LSTM_Forecasting"
)

lstm_runs = mlflow.search_runs(
    experiment_names=["Rossmann_LSTM_Forecasting"]
)

display(
    lstm_runs[
        [
            "tags.mlflow.runName",
            "metrics.rolling_one_step_mae",
            "metrics.rolling_one_step_rmse",
            "metrics.rolling_one_step_r2",
            "metrics.recursive_mae",
            "metrics.recursive_rmse",
            "metrics.recursive_r2",
            "status"
        ]
    ]
)

,tags.mlflow.runName,metrics.rolling_one_step_mae,metrics.rolling_one_step_rmse,metrics.rolling_one_step_r2,metrics.recursive_mae,metrics.recursive_rmse,metrics.recursive_r2,status
0,Final_Multivariate_LSTM,843193.42,1470030.87,0.7927,857818.27,1475753.09,0.7911,FINISHED
1,Final_Multivariate_LSTM,843193.42,1470030.87,0.7927,857818.27,1475753.09,0.7911,FINISHED


## Model Registration – Sales and Customers

In [28]:
import mlflow
import pandas as pd

from mlflow.tracking import MlflowClient

from pathlib import Path

PROJECT_DIR = Path.cwd().parent

MLFLOW_DB_PATH = PROJECT_DIR / "mlflow.db"

tracking_uri = f"sqlite:///{MLFLOW_DB_PATH.as_posix()}"

mlflow.set_tracking_uri(tracking_uri)
mlflow.set_registry_uri(tracking_uri)

client = MlflowClient()

print("Tracking URI:", mlflow.get_tracking_uri())
print("Registry URI:", mlflow.get_registry_uri())

Tracking URI: sqlite:///C:/Users/ijish/Desktop/Rossmann Sales Forecasting Project/mlflow.db
Registry URI: sqlite:///C:/Users/ijish/Desktop/Rossmann Sales Forecasting Project/mlflow.db


In [29]:
def get_final_run(experiment_name, run_name):

    runs = mlflow.search_runs(
        experiment_names=[experiment_name]
    )

    selected = runs[
        (runs["tags.mlflow.runName"] == run_name) &
        (runs["status"] == "FINISHED")
    ]

    if selected.empty:
        raise ValueError(f"Run not found: {run_name}")

    selected = selected.sort_values(
        "start_time",
        ascending=False
    )

    return selected.iloc[0]["run_id"]


sales_run_id = get_final_run(
    "Rossmann_Sales_Prediction",
    "Final_Sales_Decision Tree"
)

customers_run_id = get_final_run(
    "Rossmann_Customers_Prediction",
    "Final_Customers_Decision Tree"
)

print("Sales Run ID:", sales_run_id)
print("Customers Run ID:", customers_run_id)

Sales Run ID: 9d5a3a595e1d4747b5f18b6f7bf58aa5
Customers Run ID: 62fc3acecd724895a300e5fe43c45f16


### Register the Sales Model

In [30]:
sales_model_uri = f"runs:/{sales_run_id}/final_sales_model"

sales_registered = mlflow.register_model(
    model_uri=sales_model_uri,
    name="Rossmann_Sales_Model"
)

print("Sales model registered successfully!")
print("Version:", sales_registered.version)

Registered model 'Rossmann_Sales_Model' already exists. Creating a new version of this model...
2026/11/13 20:40:10 WARNING mlflow.tracking._model_registry.fluent: Run with id 9d5a3a595e1d4747b5f18b6f7bf58aa5 has no artifacts at artifact path 'final_sales_model', registering model based on models:/m-0f2f7ad9c7aa4332aac1dbd78c529742 instead


Sales model registered successfully!
Version: 2


Created version '2' of model 'Rossmann_Sales_Model'.


### Register the Customers Model

In [31]:
customers_model_uri = f"runs:/{customers_run_id}/final_customers_model"

customers_registered = mlflow.register_model(
    model_uri=customers_model_uri,
    name="Rossmann_Customers_Model"
)

print("Customers model registered successfully!")
print("Version:", customers_registered.version)

Registered model 'Rossmann_Customers_Model' already exists. Creating a new version of this model...
2026/11/13 20:40:11 WARNING mlflow.tracking._model_registry.fluent: Run with id 62fc3acecd724895a300e5fe43c45f16 has no artifacts at artifact path 'final_customers_model', registering model based on models:/m-fad29d8297fc4f0a86fc3aee122f806f instead


Customers model registered successfully!
Version: 2


Created version '2' of model 'Rossmann_Customers_Model'.


## Assign Champion Alias to Final Models

In [32]:
client.set_registered_model_alias(
    name="Rossmann_Sales_Model",
    alias="champion",
    version=sales_registered.version
)

client.set_registered_model_alias(
    name="Rossmann_Customers_Model",
    alias="champion",
    version=customers_registered.version
)

print("Champion aliases assigned successfully!")

Champion aliases assigned successfully!


## Load Registered Models

In [33]:
import mlflow.sklearn

loaded_sales_model = mlflow.sklearn.load_model(
    "models:/Rossmann_Sales_Model@champion"
)

loaded_customers_model = mlflow.sklearn.load_model(
    "models:/Rossmann_Customers_Model@champion"
)

print("Sales Model Loaded:", loaded_sales_model)
print("Customers Model Loaded:", loaded_customers_model)

Sales Model Loaded: Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Store', 'DayOfWeek', 'Open',
                                                   'Promo', 'SchoolHoliday',
                                                   'CompetitionDistance',
                                                   'Promo2',
                                                   'CompetitionOpenDateMissing',
                                                   'CompetitionDistance_Original',
                                                   'Year', 'Month', 'Day',
             

### Export MLFlow Models

In [51]:
import joblib
from pathlib import Path

# Project directories
PROJECT_DIR = Path.cwd().parent
MODEL_DIR = PROJECT_DIR / "models"

MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Export Sales Model
joblib.dump(
    loaded_sales_model,
    MODEL_DIR / "sales_model.joblib"
)

# Export Customers Model
joblib.dump(
    loaded_customers_model,
    MODEL_DIR / "customers_model.joblib"
)

print("Models exported successfully!")

Models exported successfully!


In [52]:
print("Sales Model:", (MODEL_DIR / "sales_model.joblib").exists())
print("Customers Model:", (MODEL_DIR / "customers_model.joblib").exists())

Sales Model: True
Customers Model: True


In [50]:
print("Sales Model:", sales_model_path)
print("Customers Model:", customers_model_path)

print("Sales Exists:", sales_model_path.exists())
print("Customers Exists:", customers_model_path.exists())

Sales Model: C:\Users\ijish\Desktop\Rossmann Sales Forecasting Project\models\rossmann_sales_model_12-11-2026-22-49-38.pkl
Customers Model: C:\Users\ijish\Desktop\Rossmann Sales Forecasting Project\models\rossmann_customers_model_12-11-2026-22-53-37.pkl
Sales Exists: True
Customers Exists: True


In [53]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_DIR = Path.cwd().parent
DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "outputs"

# Load original datasets
train_calendar = pd.read_csv(
    DATA_DIR / "train.csv",
    dtype={"StateHoliday": "string"},
    parse_dates=["Date"]
)

test_calendar = pd.read_csv(
    DATA_DIR / "test.csv",
    dtype={"StateHoliday": "string"},
    parse_dates=["Date"]
)

# Combine holiday information
calendar_df = pd.concat(
    [
        train_calendar[["Date", "StateHoliday"]],
        test_calendar[["Date", "StateHoliday"]]
    ],
    ignore_index=True
)

# Extract state holiday dates
holiday_dates = calendar_df.loc[
    calendar_df["StateHoliday"].notna()
    & (calendar_df["StateHoliday"].astype(str) != "0"),
    "Date"
].drop_duplicates()

# Add the additional holiday used in your notebook
holiday_dates = pd.concat([
    holiday_dates,
    pd.Series([pd.Timestamp("2015-10-03")])
])

holiday_dates = pd.to_datetime(
    holiday_dates
).drop_duplicates().sort_values()

# Save calendar
holiday_df = pd.DataFrame({
    "Date": holiday_dates
})

holiday_df.to_csv(
    OUTPUT_DIR / "holiday_dates.csv",
    index=False
)

print("Holiday calendar saved successfully!")
print("Total holidays:", len(holiday_df))
print(holiday_df.tail())

Holiday calendar saved successfully!
Total holidays: 39
              Date
86970   2015-05-14
74705   2015-05-25
63555   2015-06-04
1045465 2015-08-15
0       2015-10-03


## Evaluation of Registered Models Using MLFlow

In [39]:
from pathlib import Path

# Project directories
PROJECT_DIR = Path.cwd().parent

DATA_DIR = PROJECT_DIR / "data"

OUTPUT_DIR = PROJECT_DIR / "outputs"

MODEL_DIR = PROJECT_DIR / "models"

print("Project Directory:", PROJECT_DIR)
print("Data Directory:", DATA_DIR)
print("Output Directory:", OUTPUT_DIR)

Project Directory: C:\Users\ijish\Desktop\Rossmann Sales Forecasting Project
Data Directory: C:\Users\ijish\Desktop\Rossmann Sales Forecasting Project\data
Output Directory: C:\Users\ijish\Desktop\Rossmann Sales Forecasting Project\outputs


In [41]:
train_df = pd.read_csv(
    OUTPUT_DIR / "train_merged.csv",
    parse_dates=["Date"]
)

print("Dataset Shape:", train_df.shape)

Dataset Shape: (1017209, 20)


In [42]:
# Make a copy of the original dataset
train_df = train_df.copy()

# Ensure Date is datetime
train_df["Date"] = pd.to_datetime(train_df["Date"])

# Sort by date
train_df = train_df.sort_values("Date").reset_index(drop=True)


# --------------------------------
# 1. Date-based features
# --------------------------------

train_df["Year"] = train_df["Date"].dt.year
train_df["Month"] = train_df["Date"].dt.month
train_df["Day"] = train_df["Date"].dt.day

train_df["DayOfWeek"] = train_df["Date"].dt.dayofweek

train_df["IsWeekend"] = (
    train_df["DayOfWeek"].isin([5, 6]).astype(int)
)

train_df["WeekOfYear"] = (
    train_df["Date"].dt.isocalendar().week.astype(int)
)

train_df["Quarter"] = train_df["Date"].dt.quarter

train_df["MonthPeriod"] = np.select(
    [
        train_df["Day"] <= 10,
        train_df["Day"].between(11, 20)
    ],
    [
        "Beginning",
        "Middle"
    ],
    default="End"
)

train_df["IsMonthStart"] = (
    train_df["Date"].dt.is_month_start.astype(int)
)

train_df["IsMonthEnd"] = (
    train_df["Date"].dt.is_month_end.astype(int)
)

train_df["IsYearStart"] = (
    train_df["Date"].dt.is_year_start.astype(int)
)

train_df["IsYearEnd"] = (
    train_df["Date"].dt.is_year_end.astype(int)
)


# --------------------------------
# 2. State holiday distance features
# --------------------------------

# Look for the original test dataset
possible_test_paths = [
    PROJECT_DIR / "data" / "test.csv",
    DATA_DIR / "test.csv"
]

test_path = next(
    (p for p in possible_test_paths if p.exists()),
    None
)

# Use dates from train and test to build the holiday calendar
holiday_frames = [
    train_df[["Date", "StateHoliday"]].copy()
]

if test_path is not None:
    test_calendar = pd.read_csv(
        test_path,
        parse_dates=["Date"]
    )

    holiday_frames.append(
        test_calendar[["Date", "StateHoliday"]]
    )
else:
    print(
        "WARNING: test.csv not found. "
        "Holiday distances may be incomplete."
    )

holiday_df = pd.concat(
    holiday_frames,
    ignore_index=True
)

holiday_df["StateHoliday"] = (
    holiday_df["StateHoliday"].astype(str).str.strip()
)

holiday_dates = np.sort(
    holiday_df.loc[
        holiday_df["StateHoliday"].notna()
        & ~holiday_df["StateHoliday"].isin(["0", "nan", "None", ""]),
        "Date"
    ].dropna().unique()
)

# Calculate distance to next holiday
date_values = train_df["Date"].values

next_idx = np.searchsorted(
    holiday_dates,
    date_values,
    side="left"
)

days_to_holiday = np.full(
    len(train_df),
    np.nan
)

valid_next = next_idx < len(holiday_dates)

days_to_holiday[valid_next] = (
    holiday_dates[next_idx[valid_next]] -
    date_values[valid_next]
) / np.timedelta64(1, "D")

train_df["Days_To_StateHoliday"] = days_to_holiday


# Calculate distance from previous holiday
previous_idx = (
    np.searchsorted(
        holiday_dates,
        date_values,
        side="right"
    ) - 1
)

days_after_holiday = np.full(
    len(train_df),
    np.nan
)

valid_previous = previous_idx >= 0

days_after_holiday[valid_previous] = (
    date_values[valid_previous] -
    holiday_dates[previous_idx[valid_previous]]
) / np.timedelta64(1, "D")

train_df["Days_After_StateHoliday"] = days_after_holiday


# --------------------------------
# 3. Competition features
# --------------------------------

competition_year = pd.to_numeric(
    train_df["CompetitionOpenSinceYear"],
    errors="coerce"
)

competition_month = pd.to_numeric(
    train_df["CompetitionOpenSinceMonth"],
    errors="coerce"
)

train_df["CompetitionOpenMonths"] = (
    (train_df["Year"] - competition_year) * 12
    + (train_df["Month"] - competition_month)
)

# Identify missing competition information
train_df["CompetitionMonths_Missing"] = (
    train_df["CompetitionOpenMonths"].isna().astype(int)
)


# --------------------------------
# 4. Promo2 active feature
# --------------------------------

promo2 = pd.to_numeric(
    train_df["Promo2"],
    errors="coerce"
).fillna(0)

promo_year = pd.to_numeric(
    train_df["Promo2SinceYear"],
    errors="coerce"
)

promo_week = pd.to_numeric(
    train_df["Promo2SinceWeek"],
    errors="coerce"
)

iso_dates = train_df["Date"].dt.isocalendar()

current_year_week = (
    iso_dates["year"].astype(int) * 100
    + iso_dates["week"].astype(int)
)

promo_start_year_week = (
    promo_year.fillna(0) * 100
    + promo_week.fillna(0)
)

promo_month = train_df["Date"].dt.strftime("%b").str.lower()

promo_interval = (
    train_df["PromoInterval"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.split(",")
)

month_in_interval = [
    month in interval
    for month, interval in zip(promo_month, promo_interval)
]

train_df["Promo2Active"] = (
    (promo2 == 1)
    & promo_year.notna()
    & promo_week.notna()
    & (current_year_week >= promo_start_year_week)
    & np.array(month_in_interval)
).astype(int)


print("Feature engineering completed!")

print("Updated dataset shape:", train_df.shape)

print("\nNewly created features:")
print([
    "Year", "Month", "Day", "IsWeekend",
    "WeekOfYear", "Quarter", "MonthPeriod",
    "IsMonthStart", "IsMonthEnd", "IsYearStart",
    "IsYearEnd", "Days_To_StateHoliday",
    "Days_After_StateHoliday", "CompetitionOpenMonths",
    "Promo2Active", "CompetitionMonths_Missing"
])

Feature engineering completed!
Updated dataset shape: (1017209, 36)

Newly created features:
['Year', 'Month', 'Day', 'IsWeekend', 'WeekOfYear', 'Quarter', 'MonthPeriod', 'IsMonthStart', 'IsMonthEnd', 'IsYearStart', 'IsYearEnd', 'Days_To_StateHoliday', 'Days_After_StateHoliday', 'CompetitionOpenMonths', 'Promo2Active', 'CompetitionMonths_Missing']


In [43]:
# Sort chronologically
train_df = train_df.sort_values("Date").reset_index(drop=True)

# Six-week validation period
validation_start = (
    train_df["Date"].max() - pd.Timedelta(weeks=6)
)

train_part = train_df[
    train_df["Date"] < validation_start
].copy()

validation_part = train_df[
    train_df["Date"] >= validation_start
].copy()

print("Training shape:", train_part.shape)
print("Validation shape:", validation_part.shape)

print("Validation start:", validation_part["Date"].min())
print("Validation end:", validation_part["Date"].max())

Training shape: (969264, 36)
Validation shape: (47945, 36)
Validation start: 2015-06-19 00:00:00
Validation end: 2015-07-31 00:00:00


In [44]:
# Columns excluded during original model training
drop_cols = [
    "Sales",
    "Customers",
    "Id",
    "Date",
    "PromoInterval",
    "CompetitionOpenSinceMonth",
    "CompetitionOpenSinceYear",
    "Promo2SinceWeek",
    "Promo2SinceYear"
]

# Prepare validation features
X_val = validation_part.drop(
    columns=drop_cols,
    errors="ignore"
)

# Get expected model features
sales_features = list(
    loaded_sales_model.feature_names_in_
)

customer_features = list(
    loaded_customers_model.feature_names_in_
)

# Check for any remaining missing columns
missing_sales = set(sales_features) - set(X_val.columns)

missing_customers = (
    set(customer_features) - set(X_val.columns)
)

print("Missing Sales features:", missing_sales)
print("Missing Customers features:", missing_customers)

if missing_sales or missing_customers:
    raise ValueError(
        "Some required model features are still missing. "
        "Check the original feature engineering."
    )

# Align features in the exact training order
X_val_sales = X_val[sales_features].copy()

X_val_customers = X_val[customer_features].copy()

# Actual values
y_sales_val = validation_part["Sales"]

y_customers_val = validation_part["Customers"]

print("\nSales features:", X_val_sales.shape)
print("Customers features:", X_val_customers.shape)

Missing Sales features: set()
Missing Customers features: set()

Sales features: (47945, 28)
Customers features: (47945, 28)


### Test and Evaluate the Registered Models

In [45]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# Load registered models and predict
sales_predictions = loaded_sales_model.predict(X_val_sales)

customer_predictions = loaded_customers_model.predict(
    X_val_customers
)


# Evaluation function
def evaluate_model(y_actual, y_pred, model_name):

    mae = mean_absolute_error(y_actual, y_pred)

    rmse = np.sqrt(
        mean_squared_error(y_actual, y_pred)
    )

    r2 = r2_score(y_actual, y_pred)

    return {
        "Model": model_name,
        "MAE": mae,
        "RMSE": rmse,
        "R2 Score": r2
    }


# Evaluate both models
evaluation_results = []

evaluation_results.append(
    evaluate_model(
        y_sales_val,
        sales_predictions,
        "Sales Decision Tree"
    )
)

evaluation_results.append(
    evaluate_model(
        y_customers_val,
        customer_predictions,
        "Customers Decision Tree"
    )
)

evaluation_df = pd.DataFrame(evaluation_results)

display(evaluation_df.round(3))

,Model,MAE,RMSE,R2 Score
0,Sales Decision Tree,971.099,1596.688,0.815
1,Customers Decision Tree,93.100,187.224,0.818


In [46]:
print("Validation period:")
print(validation_part["Date"].min())
print(validation_part["Date"].max())

print("\nValidation rows:", len(validation_part))

print("\nSales features:")
print(X_val_sales.columns.tolist())

print("\nCustomers features:")
print(X_val_customers.columns.tolist())

Validation period:
2015-06-19 00:00:00
2015-07-31 00:00:00

Validation rows: 47945

Sales features:
['Store', 'DayOfWeek', 'Open', 'Promo', 'StateHoliday', 'SchoolHoliday', 'StoreType', 'Assortment', 'CompetitionDistance', 'Promo2', 'CompetitionOpenDateMissing', 'CompetitionDistance_Original', 'Year', 'Month', 'Day', 'IsWeekend', 'WeekOfYear', 'Quarter', 'MonthPeriod', 'IsMonthStart', 'IsMonthEnd', 'IsYearStart', 'IsYearEnd', 'Days_To_StateHoliday', 'Days_After_StateHoliday', 'CompetitionOpenMonths', 'Promo2Active', 'CompetitionMonths_Missing']

Customers features:
['Store', 'DayOfWeek', 'Open', 'Promo', 'StateHoliday', 'SchoolHoliday', 'StoreType', 'Assortment', 'CompetitionDistance', 'Promo2', 'CompetitionOpenDateMissing', 'CompetitionDistance_Original', 'Year', 'Month', 'Day', 'IsWeekend', 'WeekOfYear', 'Quarter', 'MonthPeriod', 'IsMonthStart', 'IsMonthEnd', 'IsYearStart', 'IsYearEnd', 'Days_To_StateHoliday', 'Days_After_StateHoliday', 'CompetitionOpenMonths', 'Promo2Active', 'Compe

In [48]:
print("Validation start:", validation_part["Date"].min())
print("Validation end:", validation_part["Date"].max())

print("Validation rows:", len(validation_part))

print("Sales target statistics:")
print(y_sales_val.describe())

print("\nCustomers target statistics:")
print(y_customers_val.describe())

Validation start: 2015-06-19 00:00:00
Validation end: 2015-07-31 00:00:00
Validation rows: 47945
Sales target statistics:
count    47945.000000
mean      6039.665241
std       3710.997036
min          0.000000
25%       4134.000000
50%       5945.000000
75%       7994.000000
max      41551.000000
Name: Sales, dtype: float64

Customers target statistics:
count    47945.000000
mean       634.004338
std        438.376298
min          0.000000
25%        432.000000
50%        603.000000
75%        808.000000
max       4783.000000
Name: Customers, dtype: float64


In [49]:
print("Sales feature count:", len(X_val_sales.columns))
print("Customers feature count:", len(X_val_customers.columns))

print("\nSales model expected features:")
print(list(loaded_sales_model.feature_names_in_))

print("\nCustomers model expected features:")
print(list(loaded_customers_model.feature_names_in_))

Sales feature count: 28
Customers feature count: 28

Sales model expected features:
['Store', 'DayOfWeek', 'Open', 'Promo', 'StateHoliday', 'SchoolHoliday', 'StoreType', 'Assortment', 'CompetitionDistance', 'Promo2', 'CompetitionOpenDateMissing', 'CompetitionDistance_Original', 'Year', 'Month', 'Day', 'IsWeekend', 'WeekOfYear', 'Quarter', 'MonthPeriod', 'IsMonthStart', 'IsMonthEnd', 'IsYearStart', 'IsYearEnd', 'Days_To_StateHoliday', 'Days_After_StateHoliday', 'CompetitionOpenMonths', 'Promo2Active', 'CompetitionMonths_Missing']

Customers model expected features:
['Store', 'DayOfWeek', 'Open', 'Promo', 'StateHoliday', 'SchoolHoliday', 'StoreType', 'Assortment', 'CompetitionDistance', 'Promo2', 'CompetitionOpenDateMissing', 'CompetitionDistance_Original', 'Year', 'Month', 'Day', 'IsWeekend', 'WeekOfYear', 'Quarter', 'MonthPeriod', 'IsMonthStart', 'IsMonthEnd', 'IsYearStart', 'IsYearEnd', 'Days_To_StateHoliday', 'Days_After_StateHoliday', 'CompetitionOpenMonths', 'Promo2Active', 'Compet

### Final Observation and Conclusion
After registering the final Sales and Customers Decision Tree models in MLflow, both models were successfully loaded from the Model Registry and evaluated on the validation dataset covering **19 June 2015 to 31 July 2015**, containing **47,945 records**.

**Observations:**

- The Sales Decision Tree achieved an R² score of **0.815**, indicating that it explains approximately 81.5% of the variation in sales.
- The Customers Decision Tree achieved an R² score of **0.818**, explaining approximately 81.8% of the variation in customer numbers.
- The Sales model recorded an MAE of **971.099**, while the Customers model recorded an MAE of **93.100**.
- Both registered models successfully generated predictions, demonstrating that the saved pipelines can be loaded and used for inference.

**Conclusion:**

The MLflow Model Registry successfully stores and provides access to the final Sales and Customers models. The evaluation confirms that both registered models are operational. However, their metrics differ from the original model evaluation, so further validation is needed to confirm that the same data split and feature-engineering process were used.